# Fine-Tuning **SmolLM-135M** model for Generating Sports News:
 
 - Utilized **AG News Dataset** for a generative task using the **Sports** news available in it.
    

In [1]:
# Importing Libraries:

import numpy as np
import pandas as pd
import torch

import datasets
from datasets import load_dataset

import transformers

# Preprocessing:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling

# Training:
from transformers import TrainingArguments, Trainer

# Post Training Analysis:
from transformers import pipeline
import evaluate
import re

/home/ashish-ml-prep/Music/Personal_Videos_(Phone)/Projects/Self_Projects/ft_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading **AG News** Dataset:


In [2]:
news_dataset = load_dataset("ag_news")
news_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
# Training Dataset and features:
news_train_dataset = news_dataset["train"]
print(news_train_dataset.features)


{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


In [4]:
# Sports News:
news_train_dataset[1300]

{'text': "Phelps's chase of Spitz mark? It's history This was the event Michael Phelps didn't really need to compete in if his goal was to win eight golds. He probably would have had a better chance somewhere else.",
 'label': 1}

In [5]:
# Defining news id to label map:
news_id_to_label_map = { 0:"World", 1:"Sports", 2:"Business", 3:"Sci/Tech" }

In [6]:
# Filtering the sports news dataset using label_id:
sports_datasets = news_dataset.filter(lambda example: example["label"] == 1)
sports_datasets = sports_datasets.remove_columns("label")

## Preprocessing:

### Loading the tokenizer for SmolLM-135M:


In [7]:
# Loading tokenizer for SmolLM-135M: 
model_name = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [8]:
# We need to specify as SmolLM's tokenizer doesn't include the padding token:
tokenizer.pad_token = ( tokenizer.eos_token )  

In [9]:
# Define Tokenizer wrapper function:
def tokenizer_wrapper(batch):
    return tokenizer(batch["text"], truncation=True)


### Tokenizing the Sports News Dataset:

In [10]:
# Note: We require input_ids and attention_mask
# Since we can straight up work with token ids:

tokenized_sports_news_datasets = sports_datasets.map(
                                    tokenizer_wrapper,  
                                    batched = True, 
                                    remove_columns = ["text"],  
                                )


In [11]:
tokenized_sports_news_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1900
    })
})

In [12]:
# Showing example tokenization:
ip_index = 69
example_tokenized = tokenized_sports_news_datasets['train'][ip_index]
example_tokenized_input_ids = list(example_tokenized['input_ids'])
example_tokenized_attention_mask = list(example_tokenized['attention_mask'])

# Showing example tokenization:
print(f"tokenized_ids:\n{example_tokenized_input_ids}")
print(f"\n\nattention_mask:\n{example_tokenized_attention_mask}")



tokenized_ids:
[44745, 917, 8992, 18968, 370, 216, 35, 29, 33, 288, 48750, 31732, 534, 365, 3872, 25, 6594, 731, 41710, 8581, 12713, 3917, 582, 1658, 281, 2976, 7954, 616, 327, 650, 808, 4726, 281, 3920, 253, 3531, 284, 4573, 4653, 10463, 2994, 253, 1296, 29, 10521, 24190, 282, 260, 11554, 18968, 370, 351, 253, 216, 35, 29, 33, 9970, 10528, 30]


attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Training the Model for Fine-Tuning:


In [13]:
# Identifying device to train on GPU:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
# The parameter `mlm` ==> masked language modeling
# Since we are doing Causal Learning, we set:
# mlm = False

# Initialising the data collator:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [15]:
# Reviewing shape of the tokenized texts inputs:
samples = [tokenized_sports_news_datasets["train"][i] for i in range(5)]

for sample in samples:
    print(f"input_ids shape: {len(sample['input_ids'])}")

input_ids shape: 103
input_ids shape: 81
input_ids shape: 60
input_ids shape: 65
input_ids shape: 51


In [16]:
# Reviewing shape of the tokenized samples post using data collator:
data_collator_samples_output = data_collator(samples)
for key in data_collator_samples_output:
    print(f"{key} shape: {data_collator_samples_output[key].shape}")

input_ids shape: torch.Size([5, 103])
attention_mask shape: torch.Size([5, 103])
labels shape: torch.Size([5, 103])


### Loading the SmolLM-135M model:

In [17]:
# Loading the model (SmolLM-135M) for causal learning:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

### Setting up Training Arguments and Initialising the Trainer:

In [18]:
# Setting the Training Arguments:
batch_size = 3

training_args = TrainingArguments(
    "sports-news-generator",
    push_to_hub = False,
    per_device_train_batch_size = batch_size,
    weight_decay = 0.1,
    lr_scheduler_type = "cosine",
    learning_rate = 5e-4,
    num_train_epochs = 2,
    eval_strategy = "steps",
    eval_steps = 200,
    logging_steps = 200,
    gradient_accumulation_steps=3,
    warmup_steps=500)

In [19]:
# Shuffling dataset to pick 20000 examples to Train/Fine-Tune over:
shuffled_dataset = tokenized_sports_news_datasets["train"].shuffle(seed = 69)
training_subset_data = shuffled_dataset.select(range(20000))
eval_subset_data = tokenized_sports_news_datasets["test"].select(range(1600))

In [20]:
# Initialize the Trainer:
trainer = Trainer(
    model = model,
    tokenizer = tokenizer,
    args = training_args,
    data_collator = data_collator,
    train_dataset = training_subset_data,
    eval_dataset = eval_subset_data,
)

/tmp/ipykernel_8054/2525527858.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start Training/Fine-Tuning the model:

In [21]:
# Train the model:
trainer.train()

Step,Training Loss,Validation Loss
200,3.745000,3.573436
400,3.476500,3.535896
600,3.456600,3.497085
800,3.376500,3.400172
1000,3.263800,3.325616
1200,3.206800,3.247756
1400,3.180900,3.197694
1600,3.102300,3.136323
1800,3.062200,3.082736
2000,2.977800,3.028889


TrainOutput(global_step=4444, training_loss=2.6970218758020823, metrics={'train_runtime': 1565.0844, 'train_samples_per_second': 25.558, 'train_steps_per_second': 2.839, 'total_flos': 1763381579953536.0, 'train_loss': 2.6970218758020823, 'epoch': 1.9994000299985002})

### Saving the Fine-Tuned model:

In [22]:
# Saving the Fine-Tuned model in './Sports_News_Generation_model' directory:
trainer.save_model("./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_V1")


## Post Training/Fine-Tuning Analysis: 

In [49]:
# Initialising pipeline for inferences:
pipe = pipeline(
    "text-generation",
    model="./Sports_News_Generation_model/sports_news_gen_model_bs2_ep3_FT_V1", 
    device=device,
)



# Test generation:
print(
    pipe("LA Lakers", do_sample=True, temperature=0.8, max_new_tokens=30)[0][
        "generated_text"
    ]
)

Device set to use cuda


LA Lakers 44, Lakers 27 Rashard Lewis scored 27 points and grabbed seven rebounds to lead the Los Angeles Lakers over


In [50]:
# Example generation:
input_prompt_example_1 = "Micheal Phelps wins"
generated_example_1 = pipe(input_prompt_example_1, do_sample=True, temperature=0.7, max_new_tokens=30)[0][
                        "generated_text"
                    ]

input_prompt_example_2 = "Sacramento Kings player"
generated_example_2 = pipe(input_prompt_example_2, do_sample=True, temperature=0.7, max_new_tokens=30)[0][
                        "generated_text"
                    ]


print(f"input_prompt_example_1:\n{input_prompt_example_1}\n\ngenerated_example_1:\n{generated_example_1} ")
print(f"input_prompt_example_2:\n{input_prompt_example_2}\n\ngenerated_example_2:\n{generated_example_2} ")

input_prompt_example_1:
Micheal Phelps wins

generated_example_1:
Micheal Phelps wins Olympic gold ATHENS - Michael Phelps captured his fourth Olympic gold medal of the Athens Games with a two-stroke victory over his former team- 
input_prompt_example_2:
Sacramento Kings player

generated_example_2:
Sacramento Kings player arrested, accused of assault CBC SPORTS ONLINE - Sacramento Kings player Kobe Bryant was arrested on charges he allegedly went public with videotapes 


In [51]:
# Dataset for post training analysis:
# Using 300 samples from the tokenized_sports_news_datasets for post training analysis:
test_subset = tokenized_sports_news_datasets["test"].select(range(1600, 1900)).shuffle(seed = 71)
test_subset

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 300
})

In [52]:
# Function to decode input_ids to text
def decode_input_ids(input_ids):
    return tokenizer.decode(input_ids, skip_special_tokens=True)

# Function to split the text into:
# - prompt (first line of the text) and
# - the rest as reference

def get_first_sentence(text):
    # sentences = re.split(r'(?<=[.!?])\s+', text)
    return " ".join(text.split()[:10]), text



In [53]:
# Generating prompt-reference dataset: 
prompt_reference_data = []

for elem in test_subset:
    text = decode_input_ids(elem["input_ids"])
    prompt, reference = get_first_sentence(text)
    prompt_reference_data.append({"prompt": prompt, "reference": reference})


In [54]:
# Testing 5 samples:
for i in range(5):
    print(f"Sample {i + 1}:")
    print(f"Prompt: {prompt_reference_data[i]['prompt']}")
    print(f"Reference: {prompt_reference_data[i]['reference']}\n")


Sample 1:
Prompt: Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati
Reference: Bengals' Palmer Questionable for Sunday  CINCINNATI (Sports Network) - Cincinnati Bengals  quarterback Carson Palmer is questionable for Sunday's game  against Buffalo after an MRI exam Monday revealed no serious  damage to his left knee.

Sample 2:
Prompt: Expectations too lofty for unlucky Willingham Three seasons after hiring
Reference: Expectations too lofty for unlucky Willingham Three seasons after hiring Tyrone Willingham as head coach of the football program, the powers that be in South Bend, Ind., fired the 28-year coaching veteran Tuesday, one month prior to the Fighting Irish #39;s scheduled matchup with UCLA in the Insight Bowl 

Sample 3:
Prompt: Red Sox Formula Is a Model for Success As the
Reference: Red Sox Formula Is a Model for Success As the shuffling of players intensifies this off-season, some of the Boston Red Sox' pictures will come down. The champions wi

In [55]:
# Declaring pipe for text generation:
pipe = pipeline("text-generation", model=model_name, device=device)


Device set to use cuda


In [56]:
# Prompt Texts:
prompts_texts = [data["prompt"] for data in prompt_reference_data]


In [57]:
# Generating completions:
# generated_texts = [pipe(data["prompt"], do_sample=True, temperature=0.7, max_new_tokens=42)[0]["generated_text"] for data in prompt_reference_data[:5]]
generated_texts_list = pipe(prompts_texts, do_sample=True, temperature=1, max_new_tokens=42)
# generated_texts_list = generated_texts_list[1:]

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for o

In [58]:
generated_texts_list[:5]

[[{'generated_text': "Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati\nPossibly: 44.01775513731, -68.7520689798, -23.423"}],
 [{'generated_text': 'Expectations too lofty for unlucky Willingham Three seasons after hiring this guy, but they made some cool points about it’s a good deal on what it was actually like living in Pellston.\nThe original story of the film was a story about how the movie'}],
 [{'generated_text': 'Red Sox Formula Is a Model for Success As the American Cup’s 2017 winners were announced, they also revealed which team had won the league’s second Super Bowl. The two were: the New York Knicks, by six runs'}],
 [{'generated_text': 'George sits for first time in career Irving, TX (Sports) / History)\nWhat was the first baseball game between the Yankees and the White Sox?\n1882; The Yankees played in Pittsburgh, PA, and the White S'}],
 [{'generated_text': 'Poll cost us victory - Cech Referee Graham Poll came in first in the 2002 Cildescor

In [59]:
generated_texts = [generated_text[0]["generated_text"] for generated_text in generated_texts_list]


### Calculating ROUGE and BLEU Scores:

In [60]:
# Loading BLEU and ROUGE scores from evaluate:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [61]:
# Calculating BLEU scores and ROUGE scores for all 300 samples:
bleu_scores = [bleu.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]
rouge_scores = [rouge.compute(predictions=[generated], references=[data["reference"]]) for data, generated in zip(prompt_reference_data, generated_texts)]


In [62]:
# Printing BLEU and ROUGE scores for first 5 samples:
print(f"BLEU Scores for the first 5 samples: {bleu_scores[:5]}\n")
print(f"ROUGE Scores for the first 5 samples: {rouge_scores[:5]}")


BLEU Scores for the first 5 samples: [{'bleu': 0.23180269455467872, 'precisions': [0.631578947368421, 0.6111111111111112, 0.5882352941176471, 0.5625], 'brevity_penalty': 0.38776010329632493, 'length_ratio': 0.5135135135135135, 'translation_length': 19, 'reference_length': 37}, {'bleu': 0.16569640461972113, 'precisions': [0.3404255319148936, 0.21739130434782608, 0.17777777777777778, 0.1590909090909091], 'brevity_penalty': 0.7746692236459022, 'length_ratio': 0.7966101694915254, 'translation_length': 47, 'reference_length': 59}, {'bleu': 0.23242860695451686, 'precisions': [0.3333333333333333, 0.24390243902439024, 0.2, 0.1794871794871795], 'brevity_penalty': 1.0, 'length_ratio': 1.2, 'translation_length': 42, 'reference_length': 35}, {'bleu': 0.24897336736896636, 'precisions': [0.45454545454545453, 0.27906976744186046, 0.23809523809523808, 0.21951219512195122], 'brevity_penalty': 0.8725252928694237, 'length_ratio': 0.88, 'translation_length': 44, 'reference_length': 50}, {'bleu': 0.2048614

### For Overall BLEU and ROUGE scores:

In [63]:
prompts_texts_bleu = [[elem['prompt']] for elem in prompt_reference_data]
prompts_texts_bleu

[["Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati"],
 ['Expectations too lofty for unlucky Willingham Three seasons after hiring'],
 ['Red Sox Formula Is a Model for Success As the'],
 ['George sits for first time in career Irving, TX (Sports'],
 ['Poll cost us victory - Cech Referee Graham Poll came'],
 ['Oxford 18 Cambridge 11 Replacement winger Ross Lavery scored the'],
 ['Barca wins again Barcelona has moved 12 points clear at'],
 ['The Newest Hope ; Marriage of Necessity Just Might Work'],
 ['Football Association charges Bolton striker over spitting incident Bolton striker'],
 ['ITA: Juventus blows 2-goal lead againt Inter Milan Serie A'],
 ['Milan Mandaric statement quot;Over the past two-and-a-half years the football'],
 ['Sherman remains confident after Packers #39; flop in Philly They'],
 ['Utah Hires Whittingham to Replace Meyer (AP) AP - Utah'],
 ['Memphis indefinitely suspends Sean Banks Memphis forward Sean Banks was'],
 ['Delaney keen to 

In [64]:
# Get reference texts for the generated output:
reference_texts = [[elem['reference']] for elem in prompt_reference_data]
reference_texts[:5]


[["Bengals' Palmer Questionable for Sunday  CINCINNATI (Sports Network) - Cincinnati Bengals  quarterback Carson Palmer is questionable for Sunday's game  against Buffalo after an MRI exam Monday revealed no serious  damage to his left knee."],
 ['Expectations too lofty for unlucky Willingham Three seasons after hiring Tyrone Willingham as head coach of the football program, the powers that be in South Bend, Ind., fired the 28-year coaching veteran Tuesday, one month prior to the Fighting Irish #39;s scheduled matchup with UCLA in the Insight Bowl '],
 ["Red Sox Formula Is a Model for Success As the shuffling of players intensifies this off-season, some of the Boston Red Sox' pictures will come down. The champions will have to change."],
 ['George sits for first time in career Irving, TX (Sports Network) - Dallas Cowboys running back Eddie George was inactive for Sunday #39;s game against New Orleans as a healthy scratch and missed a game for the first time in his NFL career.'],
 ['Pol

In [65]:
generated_texts[:5]

["Bengals' Palmer Questionable for Sunday CINCINNATI (Sports Network) - Cincinnati\nPossibly: 44.01775513731, -68.7520689798, -23.423",
 'Expectations too lofty for unlucky Willingham Three seasons after hiring this guy, but they made some cool points about it’s a good deal on what it was actually like living in Pellston.\nThe original story of the film was a story about how the movie',
 'Red Sox Formula Is a Model for Success As the American Cup’s 2017 winners were announced, they also revealed which team had won the league’s second Super Bowl. The two were: the New York Knicks, by six runs',
 'George sits for first time in career Irving, TX (Sports) / History)\nWhat was the first baseball game between the Yankees and the White Sox?\n1882; The Yankees played in Pittsburgh, PA, and the White S',
 'Poll cost us victory - Cech Referee Graham Poll came in first in the 2002 Cildescoration, followed by the 2004 PAC in the 2005 and 2006 Cildescor']

In [66]:
# Overall BLEU Score:
overall_bleu_score = bleu.compute(predictions=generated_texts, references=reference_texts)
print(f"Overall BLEU Score: {overall_bleu_score['bleu']:.4f}")


Overall BLEU Score: 0.2554


In [67]:
# ROUGE scores:
rouge_1_f1 = [score['rouge1'] for score in rouge_scores]
rouge_2_f1 = [score['rouge2'] for score in rouge_scores]
rouge_l_f1 = [score['rougeL'] for score in rouge_scores]

# Calculate average ROUGE F1 scores:
average_rouge_1_f1 = np.mean(rouge_1_f1)
average_rouge_2_f1 = np.mean(rouge_2_f1)
average_rouge_l_f1 = np.mean(rouge_l_f1)

# Print the average ROUGE scores:
print(f"Average ROUGE-1 F1: {average_rouge_1_f1}")
print(f"Average ROUGE-2 F1: {average_rouge_2_f1}")
print(f"Average ROUGE-L F1: {average_rouge_l_f1}")

Average ROUGE-1 F1: 0.363540100871332
Average ROUGE-2 F1: 0.2575817893465021
Average ROUGE-L F1: 0.34019848113889206


In [ ]:
import torch
torch.cuda.empty_cache()
